<a href="https://colab.research.google.com/github/aniketkoley-iisc/ev-gap-analysis-india/blob/main/notebooks/01_HDFS_Lakehouse_and_FullScale_Bronze_Ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#######################################
### 0.1 START INIT ENVIRONMENT

from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyspark h3 osmium requests

import os
import shutil

# Persistent Google Drive root (so multi-GB downloads are saved permanently)
DRIVE_ROOT = "/content/drive/MyDrive/EV_Gap_Project"

# Local Colab disk directories (Spark reads/writes local files much faster than Drive/FUSE)
LOCAL_DATA_DIR = "/content/data"
BRONZE_DIR = f"{LOCAL_DATA_DIR}/bronze"
SILVER_DIR = f"{LOCAL_DATA_DIR}/silver"
GOLD_DIR   = f"{LOCAL_DATA_DIR}/gold"

for d in [DRIVE_ROOT, f"{DRIVE_ROOT}/bronze", f"{DRIVE_ROOT}/silver", f"{DRIVE_ROOT}/gold",
          LOCAL_DATA_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(d, exist_ok=True)

# Clone your public GitHub repository (if not already present)
REPO_DIR = "/content/ev-gap-analysis-india"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aniketkoley-iisc/ev-gap-analysis-india.git {REPO_DIR}

print("Environment & Lakehouse directories initialized in local Colab storage and Google Drive.")

### 0.1 END INIT ENVIRONMENT

Mounted at /content/drive
Cloning into '/content/ev-gap-analysis-india'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 12 (delta 1), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 4.32 KiB | 2.16 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Environment & Lakehouse directories initialized in local Colab storage and Google Drive.


In [2]:
#######################################
### 0.2 START OF PYSPARK INIT

import os
# Clear any leftover environment variables from earlier cells so PySpark uses Colab's default Java
for var in ["JAVA_HOME", "HADOOP_HOME", "HADOOP_COMMON_HOME", "HADOOP_HDFS_HOME", "HADOOP_CONF_DIR"]:
    os.environ.pop(var, None)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType, BooleanType, TimestampType, DateType
)
from pyspark.sql import DataFrame

spark = (
    SparkSession.builder
    .appName("da231o_ev_gap_analysis_india")
    .config("spark.executor.memory", "10g")
    .config("spark.driver.memory", "10g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "10g")
    .getOrCreate()
)

print("Spark Session is ready! Spark version:", spark.version)

### 0.2 END OF PYSPARK INIT

Spark Session is ready! Spark version: 4.0.4


In [3]:
#######################################
### 0.3 START BRONZE DATA ACQUISITION (IDEMPOTENT / SAFE TO RUN MULTIPLE TIMES)

import os
import json
import shutil
import requests

# -------------------------------------------------------------------------
# 1. Full-Scale Geofabrik India OpenStreetMap (1.59 GB .osm.pbf)
# -------------------------------------------------------------------------
OSM_PBF_DRIVE = f"{DRIVE_ROOT}/bronze/india-latest.osm.pbf"
OSM_PBF_LOCAL = f"{BRONZE_DIR}/india-latest.osm.pbf"

if os.path.exists(OSM_PBF_LOCAL) and os.path.getsize(OSM_PBF_LOCAL) > 1_000_000_000:
    print("⚡ Skipping download: india-latest.osm.pbf already on local Colab disk!")
elif os.path.exists(OSM_PBF_DRIVE) and os.path.getsize(OSM_PBF_DRIVE) > 1_000_000_000:
    print("⚡ Skipping download: copying india-latest.osm.pbf (1.59 GB) from Google Drive...")
    shutil.copy2(OSM_PBF_DRIVE, OSM_PBF_LOCAL)
else:
    print("🌍 Downloading full-scale India OpenStreetMap (india-latest.osm.pbf) from Geofabrik...")
    !wget -c --progress=bar:force:noscroll "https://download.geofabrik.de/asia/india-latest.osm.pbf" -O "{OSM_PBF_LOCAL}"
    shutil.copy2(OSM_PBF_LOCAL, OSM_PBF_DRIVE)

pbf_size_gb = os.path.getsize(OSM_PBF_LOCAL) / (1024 ** 3)
print(f"✅ Geofabrik India OSM PBF Ready: {OSM_PBF_LOCAL} ({pbf_size_gb:.2f} GB)")

# -------------------------------------------------------------------------
# 2. Live All-India Charging Stations from Overpass API
# -------------------------------------------------------------------------
OVERPASS_JSON_DRIVE = f"{DRIVE_ROOT}/bronze/osm_india_charging_stations_live.json"
OVERPASS_JSON_LOCAL = f"{BRONZE_DIR}/osm_india_charging_stations_live.json"

if os.path.exists(OVERPASS_JSON_LOCAL) and os.path.getsize(OVERPASS_JSON_LOCAL) > 10_000:
    print("⚡ Skipping API call: osm_india_charging_stations_live.json already on local Colab disk!")
elif os.path.exists(OVERPASS_JSON_DRIVE) and os.path.getsize(OVERPASS_JSON_DRIVE) > 10_000:
    print("⚡ Skipping API call: copying osm_india_charging_stations_live.json from Google Drive...")
    shutil.copy2(OVERPASS_JSON_DRIVE, OVERPASS_JSON_LOCAL)
else:
    print("⚡ Querying live OpenStreetMap Overpass API for all amenity=charging_station across India...")
    overpass_query = (
        '[out:json][timeout:180];'
        'area["ISO3166-1"="IN"][admin_level=2]->.india;'
        '(node["amenity"="charging_station"](area.india);'
        'way["amenity"="charging_station"](area.india););'
        'out center tags;'
    )
    headers = {
        "User-Agent": "IISc-DA231o-EV-Gap-Project/1.0 (aniketkoley@iisc.ac.in)",
        "Accept": "application/json"
    }
    endpoints = [
        "https://lz4.overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass-api.de/api/interpreter",
    ]
    for url in endpoints:
        try:
            resp = requests.post(url, data={"data": overpass_query}, headers=headers, timeout=180)
            if resp.status_code == 200 and "elements" in resp.text:
                with open(OVERPASS_JSON_LOCAL, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                shutil.copy2(OVERPASS_JSON_LOCAL, OVERPASS_JSON_DRIVE)
                break
        except Exception as e:
            print(f"  Mirror {url} busy ({e}), trying next mirror...")

with open(OVERPASS_JSON_LOCAL, "r", encoding="utf-8") as f:
    overpass_data = json.load(f)

elements = overpass_data.get("elements", [])
print(f"✅ Live Overpass API India Charging Stations Ready: {len(elements):,} raw OSM station records!")

### 0.3 END BRONZE DATA ACQUISITION

⚡ Skipping download: copying india-latest.osm.pbf (1.59 GB) from Google Drive...
✅ Geofabrik India OSM PBF Ready: /content/data/bronze/india-latest.osm.pbf (1.59 GB)
⚡ Skipping API call: copying osm_india_charging_stations_live.json from Google Drive...
✅ Live Overpass API India Charging Stations Ready: 610 raw OSM station records!


In [4]:
#######################################
### 0.4 START DIRECT EXTERNAL FILE INGESTION & SPARK READ (NO MANUAL DATA)

import os
import json
import glob
import shutil
import osmium
import pandas as pd

# -------------------------------------------------------------------------
# 1. Clone Real External EV & Charging Station Data Repositories (Once Only)
# -------------------------------------------------------------------------
EXT_REPOS_LOCAL = f"{BRONZE_DIR}/external_sources"
EXT_REPOS_DRIVE = f"{DRIVE_ROOT}/bronze/external_sources"
os.makedirs(EXT_REPOS_LOCAL, exist_ok=True)

repos = {
    "e_amrit_chargers": "https://github.com/mglsj/indian_ev_charging_stations.git",
    "vahan_ev_market":  "https://github.com/Narendraspatil/Indian_EV_Market_Analysis.git",
    "india_ev_sales":   "https://github.com/saurabhtehare/Indian_EV_Sales_Analytics_mydashboard.git",
}

for name, url in repos.items():
    local_target = f"{EXT_REPOS_LOCAL}/{name}"
    drive_target = f"{EXT_REPOS_DRIVE}/{name}"
    if os.path.exists(local_target) and os.listdir(local_target):
        print(f"⚡ Skipping clone: {name} already on local Colab disk!")
    elif os.path.exists(drive_target) and os.listdir(drive_target):
        print(f"⚡ Copying {name} from Google Drive backup...")
        shutil.copytree(drive_target, local_target, dirs_exist_ok=True)
    else:
        print(f"🌐 Cloning real dataset repository: {name} ({url})...")
        !git clone --depth 1 "{url}" "{local_target}"
        shutil.copytree(local_target, drive_target, dirs_exist_ok=True)

# List all downloaded CSV / JSON / GeoJSON files discovered across external sources
discovered_files = (
    glob.glob(f"{EXT_REPOS_LOCAL}/**/*.csv", recursive=True) +
    glob.glob(f"{EXT_REPOS_LOCAL}/**/*.json", recursive=True) +
    glob.glob(f"{EXT_REPOS_LOCAL}/**/*.geojson", recursive=True)
)
print(f"\n📂 Discovered {len(discovered_files)} real external data files:")
for fpath in discovered_files:
    print("  ->", os.path.relpath(fpath, EXT_REPOS_LOCAL), f"({os.path.getsize(fpath)/1024:.1f} KB)")

# -------------------------------------------------------------------------
# 2. Extract Real OSM Primitives Directly from 1.59 GB india-latest.osm.pbf
# -------------------------------------------------------------------------
FILTERED_PBF_LOCAL = f"{BRONZE_DIR}/india_ev_filtered.osm.pbf"
FILTERED_PBF_DRIVE = f"{DRIVE_ROOT}/bronze/india_ev_filtered.osm.pbf"
OSM_BRONZE_LOCAL   = f"{BRONZE_DIR}/osm_primitives_bronze.parquet"
OSM_BRONZE_DRIVE   = f"{DRIVE_ROOT}/bronze/osm_primitives_bronze.parquet"

if os.path.exists(OSM_BRONZE_LOCAL):
    print("\n⚡ Skipping OSM PBF extraction: osm_primitives_bronze.parquet already on local disk!")
elif os.path.exists(OSM_BRONZE_DRIVE):
    print("\n⚡ Copying osm_primitives_bronze.parquet from Google Drive...")
    shutil.copytree(OSM_BRONZE_DRIVE, OSM_BRONZE_LOCAL, dirs_exist_ok=True)
else:
    if not (os.path.exists(FILTERED_PBF_LOCAL) and os.path.getsize(FILTERED_PBF_LOCAL) > 100_000):
        print("\n⏳ Installing osmium-tool & filtering 1.59 GB india-latest.osm.pbf...")
        !apt-get update -qq && apt-get install -y -qq osmium-tool
        !osmium tags-filter "{OSM_PBF_LOCAL}" \
            n/power=substation n/amenity=charging_station n/amenity=fuel n/amenity=parking n/highway=services n/place=city,town \
            -o "{FILTERED_PBF_LOCAL}" -f pbf,add_metadata=false --overwrite
        shutil.copy2(FILTERED_PBF_LOCAL, FILTERED_PBF_DRIVE)

    print("🔍 Reading nodes directly from filtered India OSM PBF file...")
    class DirectOSMReader(osmium.SimpleHandler):
        def __init__(self):
            super().__init__()
            self.rows = []
        def node(self, n):
            if not n.location.valid():
                return
            t = n.tags
            ftype = (
                "power=substation" if t.get("power") == "substation" else
                "amenity=charging_station" if t.get("amenity") == "charging_station" else
                "amenity=fuel" if t.get("amenity") == "fuel" else
                "amenity=parking" if t.get("amenity") == "parking" else
                "highway=services" if t.get("highway") == "services" else
                f"place={t.get('place')}" if t.get("place") in ("city", "town") else None
            )
            if ftype:
                self.rows.append({
                    "osm_id": f"node/{n.id}",
                    "feature_type": ftype,
                    "name": t.get("name", t.get("name:en", "")),
                    "state": t.get("addr:state", t.get("is_in:state", "")),
                    "operator": t.get("operator", ""),
                    "voltage": t.get("voltage", ""),
                    "capacity": t.get("capacity", ""),
                    "lat": float(n.location.lat),
                    "lon": float(n.location.lon),
                })

    reader = DirectOSMReader()
    reader.apply_file(FILTERED_PBF_LOCAL, locations=True)
    spark.createDataFrame(pd.DataFrame(reader.rows)).write.mode("overwrite").parquet(OSM_BRONZE_LOCAL)
    shutil.copytree(OSM_BRONZE_LOCAL, OSM_BRONZE_DRIVE, dirs_exist_ok=True)

# -------------------------------------------------------------------------
# 3. Directly Read the Downloaded Files Using PySpark & Inspect Schemas!
# -------------------------------------------------------------------------
osm_bronze_df = spark.read.parquet(OSM_BRONZE_LOCAL)
print("\n=== 1. OSM INDIA PRIMITIVES (Directly Read from 1.59 GB india-latest.osm.pbf) ===")
osm_bronze_df.show(5, truncate=False)
print("Total OSM Primitives rows:", osm_bronze_df.count())
osm_bronze_df.groupBy("feature_type").count().show(truncate=False)

# Directly read all discovered CSV files into Spark to inspect their raw schemas
for csv_file in sorted(glob.glob(f"{EXT_REPOS_LOCAL}/**/*.csv", recursive=True)):
    fname = os.path.basename(csv_file)
    print(f"\n=== DIRECT SPARK READ: {fname} ===")
    df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_file)
    df_raw.show(5, truncate=False)
    print(f"Rows in {fname}:", df_raw.count())
    df_raw.printSchema()

### 0.4 END DIRECT EXTERNAL FILE INGESTION & SPARK READ

⚡ Copying e_amrit_chargers from Google Drive backup...
⚡ Copying vahan_ev_market from Google Drive backup...
⚡ Copying india_ev_sales from Google Drive backup...

📂 Discovered 12 real external data files:
  -> vahan_ev_market/Dataset/dim_date.csv (0.7 KB)
  -> vahan_ev_market/Dataset/electric_vehicle_sales_by_state.csv (98.2 KB)
  -> vahan_ev_market/Dataset/electric_vehicle_sales_by_makers.csv (29.3 KB)
  -> india_ev_sales/ev_sales_by_makers_and_cat_15-24.csv (72.1 KB)
  -> india_ev_sales/OperationalPC.csv (0.5 KB)
  -> india_ev_sales/EV_Maker_with_Location.csv (3.6 KB)
  -> india_ev_sales/ev_cat_01-24.csv (13.7 KB)
  -> india_ev_sales/EV Maker by Place.csv (2.2 KB)
  -> india_ev_sales/Vehicle Class - All.csv (0.6 KB)
  -> e_amrit_chargers/data.json (63.9 KB)
  -> india_ev_sales/india_with_disputed_boundaries.geojson.geojson (10509.1 KB)
  -> e_amrit_chargers/ev_charging_stations.geojson (89.1 KB)

⚡ Copying osm_primitives_bronze.parquet from Google Drive...

=== 1. OSM INDIA PRIMITIVE

In [5]:
#######################################
### 0.5 START UNIFIED BRONZE CHARGING STATIONS PARQUET (FROM REAL JSON / GEOJSON / PBF FILES)

import os
import json
import shutil
import pandas as pd

CS_BRONZE_LOCAL = f"{BRONZE_DIR}/charging_stations_bronze.parquet"
CS_BRONZE_DRIVE = f"{DRIVE_ROOT}/bronze/charging_stations_bronze.parquet"

if os.path.exists(CS_BRONZE_LOCAL):
    print("⚡ Skipping build: charging_stations_bronze.parquet already on local Colab disk!")
elif os.path.exists(CS_BRONZE_DRIVE):
    print("⚡ Copying charging_stations_bronze.parquet from Google Drive...")
    shutil.copytree(CS_BRONZE_DRIVE, CS_BRONZE_LOCAL, dirs_exist_ok=True)
else:
    print("🔍 Directly reading the 3 real Charging Station files (E-Amrit JSON, E-Amrit GeoJSON, Live Overpass JSON)...")
    raw_stations = []

    # Source 1: e_amrit_chargers/data.json (NITI Aayog E-Amrit Portal)
    eamrit_json_path = f"{EXT_REPOS_LOCAL}/e_amrit_chargers/data.json"
    with open(eamrit_json_path, "r", encoding="utf-8") as f:
        eamrit_data = json.load(f)
    # Handle list or dict wrapper in data.json
    eamrit_list = eamrit_data if isinstance(eamrit_data, list) else eamrit_data.get("data", eamrit_data.get("features", []))
    for i, item in enumerate(eamrit_list):
        if isinstance(item, dict):
            lat = item.get("lat") or item.get("latitude") or item.get("Latitude")
            lon = item.get("lng") or item.get("lon") or item.get("longitude") or item.get("Longitude")
            if lat is not None and lon is not None:
                raw_stations.append({
                    "station_id": str(item.get("id", f"EAMRIT-JSON-{i}")),
                    "station_name": str(item.get("name") or item.get("title") or item.get("address") or "E-Amrit Charging Station"),
                    "operator": str(item.get("operator") or item.get("company") or "E-Amrit Network"),
                    "city": str(item.get("city") or item.get("district") or ""),
                    "state": str(item.get("state") or ""),
                    "lat": float(lat),
                    "lon": float(lon),
                    "capacity_raw": str(item.get("capacity") or item.get("power") or ""),
                    "source_dataset": "E_Amrit_Portal_JSON"
                })

    # Source 2: e_amrit_chargers/ev_charging_stations.geojson
    eamrit_geojson_path = f"{EXT_REPOS_LOCAL}/e_amrit_chargers/ev_charging_stations.geojson"
    with open(eamrit_geojson_path, "r", encoding="utf-8") as f:
        geojson_data = json.load(f)
    for i, feat in enumerate(geojson_data.get("features", [])):
        geom = feat.get("geometry") or {}
        props = feat.get("properties") or {}
        coords = geom.get("coordinates", [])
        if len(coords) >= 2:
            # GeoJSON standard is [longitude, latitude]
            lon, lat = float(coords[0]), float(coords[1])
            raw_stations.append({
                "station_id": str(props.get("id") or feat.get("id") or f"EAMRIT-GEO-{i}"),
                "station_name": str(props.get("name") or props.get("title") or props.get("address") or "E-Amrit GeoJSON Station"),
                "operator": str(props.get("operator") or props.get("vendor") or "E-Amrit GeoJSON"),
                "city": str(props.get("city") or ""),
                "state": str(props.get("state") or ""),
                "lat": lat,
                "lon": lon,
                "capacity_raw": str(props.get("capacity") or props.get("power") or ""),
                "source_dataset": "E_Amrit_GeoJSON"
            })

    # Source 3: Live Overpass API JSON (osm_india_charging_stations_live.json from Cell 3)
    with open(OVERPASS_JSON_LOCAL, "r", encoding="utf-8") as f:
        overpass_data = json.load(f)
    for i, el in enumerate(overpass_data.get("elements", [])):
        lat = el.get("lat") or (el.get("center") or {}).get("lat")
        lon = el.get("lon") or (el.get("center") or {}).get("lon")
        tags = el.get("tags") or {}
        if lat is not None and lon is not None:
            raw_stations.append({
                "station_id": f"OSM-OVERPASS-{el.get('id', i)}",
                "station_name": str(tags.get("name") or tags.get("operator") or "OSM Charging Station"),
                "operator": str(tags.get("operator") or tags.get("brand") or tags.get("network") or ""),
                "city": str(tags.get("addr:city") or tags.get("addr:district") or ""),
                "state": str(tags.get("addr:state") or ""),
                "lat": float(lat),
                "lon": float(lon),
                "capacity_raw": str(tags.get("socket:ccs2:output") or tags.get("rating") or tags.get("capacity") or ""),
                "source_dataset": "OSM_Live_Overpass"
            })

    # Source 4: Geofabrik 1.59 GB PBF extracted charging stations (from osm_primitives_bronze.parquet)
    pbf_chargers_pdf = (
        spark.read.parquet(OSM_BRONZE_LOCAL)
        .filter(F.col("feature_type") == "amenity=charging_station")
        .toPandas()
    )
    for _, row in pbf_chargers_pdf.iterrows():
        raw_stations.append({
            "station_id": f"OSM-PBF-{row['osm_id']}",
            "station_name": str(row["name"] or row["operator"] or "Geofabrik PBF Charger"),
            "operator": str(row["operator"] or ""),
            "city": "",
            "state": str(row["state"] or ""),
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "capacity_raw": str(row["capacity"] or row["voltage"] or ""),
            "source_dataset": "Geofabrik_India_PBF"
        })

    cs_pdf = pd.DataFrame(raw_stations)
    spark.createDataFrame(cs_pdf).write.mode("overwrite").parquet(CS_BRONZE_LOCAL)
    shutil.copytree(CS_BRONZE_LOCAL, CS_BRONZE_DRIVE, dirs_exist_ok=True)

# Inspect our unified Multi-Source Charging Stations Bronze Parquet table!
cs_bronze_df = spark.read.parquet(CS_BRONZE_LOCAL)
print("\n=== UNIFIED MULTI-SOURCE CHARGING STATIONS BRONZE PARQUET ===")
cs_bronze_df.show(10, truncate=False)
print("Total Raw Charging Station Records Across All Real Sources:", cs_bronze_df.count())
cs_bronze_df.groupBy("source_dataset").count().show(truncate=False)
cs_bronze_df.printSchema()

### 0.5 END UNIFIED BRONZE CHARGING STATIONS PARQUET

🔍 Directly reading the 3 real Charging Station files (E-Amrit JSON, E-Amrit GeoJSON, Live Overpass JSON)...

=== UNIFIED MULTI-SOURCE CHARGING STATIONS BRONZE PARQUET ===
+------------+-------------------------------------+---------------+------------------+-------------+---------+---------+------------+---------------+
|station_id  |station_name                         |operator       |city              |state        |lat      |lon      |capacity_raw|source_dataset |
+------------+-------------------------------------+---------------+------------------+-------------+---------+---------+------------+---------------+
|EAMRIT-GEO-0|EESL Legislative Assembly Goa        |E-Amrit GeoJSON|Goa               |Goa          |15.510472|73.837222|            |E_Amrit_GeoJSON|
|EAMRIT-GEO-1|EESL R block GK1                     |E-Amrit GeoJSON|Delhi             |Delhi        |28.547217|77.244324|            |E_Amrit_GeoJSON|
|EAMRIT-GEO-2|EESL SDA market                      |E-Amrit GeoJSON|Delhi 